In [ ]:
!pip install pandas numpy matplotlib yfinance tensorflow

mambajs 0.19.13

Process pip requirements ...



Cannot install 'pandas' from PyPI because it is a binary built package that is not compatible with WASM environments. To resolve this issue, you can: 1) Try to install it from emscripten-forge instead: "!mamba install pandas" 2) If that doesn't work, it's probably that the package was not made WASM-compatible on emscripten-forge. You can either request or contribute a new recipe for that package in https://github.com/emscripten-forge/recipes 

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf
from collections import deque
import random

np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

<class 'OSError'>: Not available

In [ ]:
ticker = "MSFT"
start_date = "2018-01-01"
end_date = "2026-03-01"   # ajusta si quieres usar una fecha más reciente

data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=True)

if data.empty:
    raise ValueError("No se descargaron datos. Revisa el ticker o las fechas.")

prices = data["Close"].dropna().values.astype(np.float32)

print(f"Datos descargados: {len(prices)} precios")
data.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(data.index, data["Close"])
plt.title(f"Precio de cierre de {ticker}")
plt.xlabel("Fecha")
plt.ylabel("Precio")
plt.grid(True)
plt.show()

In [ ]:
ACTIONS = ["Buy", "Sell", "Hold"]
ACTION_SIZE = len(ACTIONS)

window_size = 200
initial_budget = 1000.0

def get_state(prices, t, budget, n_stocks, window_size=200):
    start = max(0, t - window_size + 1)
    window = prices[start:t+1]

    if len(window) < window_size:
        pad = np.full(window_size - len(window), window[0], dtype=np.float32)
        window = np.concatenate([pad, window])

    # Normalización simple del historial
    norm_window = window / (window[0] + 1e-8) - 1.0

    state = np.concatenate([
        np.array([budget / initial_budget, n_stocks], dtype=np.float32),
        norm_window.astype(np.float32)
    ])

    return state

In [ ]:
class RandomDecisionPolicy:
    def __init__(self, actions):
        self.actions = actions

    def select_action(self, state):
        return random.randint(0, len(self.actions) - 1)

    def update_q(self, *args, **kwargs):
        pass

In [ ]:
def run_simulation(policy, prices, initial_budget=1000.0, window_size=200, training=False):
    budget = initial_budget
    n_stocks = 0
    buy_points = []
    sell_points = []
    portfolio_values = []

    for t in range(window_size, len(prices) - 1):
        current_price = float(prices[t])
        next_price = float(prices[t + 1])

        state = get_state(prices, t, budget, n_stocks, window_size)
        current_portfolio = budget + n_stocks * current_price

        action = policy.select_action(state)

        if action == 0:  # Buy
            if budget >= current_price:
                budget -= current_price
                n_stocks += 1
                buy_points.append((t, current_price))

        elif action == 1:  # Sell
            if n_stocks > 0:
                budget += current_price
                n_stocks -= 1
                sell_points.append((t, current_price))

        # Hold = 2, no hace nada

        new_portfolio = budget + n_stocks * next_price
        reward = new_portfolio - current_portfolio
        next_state = get_state(prices, t + 1, budget, n_stocks, window_size)

        if training and hasattr(policy, "update_q"):
            policy.update_q(state, action, reward, next_state, done=(t == len(prices) - 2))

        portfolio_values.append(new_portfolio)

    final_portfolio = budget + n_stocks * prices[-1]
    return final_portfolio, buy_points, sell_points, portfolio_values

In [ ]:
random_policy = RandomDecisionPolicy(ACTIONS)

results = []
for _ in range(30):
    final_value, _, _, _ = run_simulation(random_policy, prices, initial_budget, window_size)
    results.append(final_value)

print("Política aleatoria")
print("Promedio portafolio final:", np.mean(results))
print("Desviación estándar:", np.std(results))

In [ ]:
# =========================
# 8. Agente DQN con Keras
# =========================
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=5000)

        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001

        self.model = self._build_model()

    def _build_model(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(self.state_size,)),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(self.action_size, activation="linear")
        ])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss="mse"
        )
        return model

    def select_action(self, state):
        if np.random.rand() < self.epsilon:
            return random.randrange(self.action_size)

        q_values = self.model.predict(state[np.newaxis, :], verbose=0)
        return int(np.argmax(q_values[0]))

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def update_q(self, state, action, reward, next_state, done=False):
        self.remember(state, action, reward, next_state, done)

        if len(self.memory) < 32:
            return

        minibatch = random.sample(self.memory, 32)

        states = []
        targets = []

        for s, a, r, s_next, d in minibatch:
            target = self.model.predict(s[np.newaxis, :], verbose=0)[0]

            if d:
                target[a] = r
            else:
                next_q = self.model.predict(s_next[np.newaxis, :], verbose=0)[0]
                target[a] = r + self.gamma * np.max(next_q)

            states.append(s)
            targets.append(target)

        self.model.fit(
            np.array(states),
            np.array(targets),
            epochs=1,
            verbose=0
        )

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

In [ ]:
state_size = window_size + 2
agent = DQNAgent(state_size, ACTION_SIZE)

episodes = 10
training_results = []

for episode in range(episodes):
    final_value, _, _, _ = run_simulation(agent, prices, initial_budget, window_size, training=True)
    training_results.append(final_value)
    print(f"Episodio {episode+1}/{episodes} - Portafolio final: {final_value:.2f} - epsilon: {agent.epsilon:.4f}")

In [ ]:
agent.epsilon = 0.0  # solo explotación
final_value, buy_points, sell_points, portfolio_values = run_simulation(
    agent, prices, initial_budget, window_size, training=False
)

print(f"Portafolio final del agente: {final_value:.2f}")
print(f"Rendimiento: {((final_value / initial_budget) - 1) * 100:.2f}%")

In [ ]:
start_eval = window_size
buy_hold_shares = int(initial_budget // prices[start_eval])
remaining_cash = initial_budget - buy_hold_shares * prices[start_eval]
buy_hold_final = remaining_cash + buy_hold_shares * prices[-1]

print(f"Buy & Hold final: {buy_hold_final:.2f}")
print(f"Diferencia agente - buy&hold: {final_value - buy_hold_final:.2f}")

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(range(len(prices)), prices, label="Precio")

if buy_points:
    x_buy, y_buy = zip(*buy_points)
    plt.scatter(x_buy, y_buy, marker="^", s=80, label="Compra")

if sell_points:
    x_sell, y_sell = zip(*sell_points)
    plt.scatter(x_sell, y_sell, marker="v", s=80, label="Venta")

plt.title(f"Señales del agente sobre {ticker}")
plt.xlabel("Tiempo")
plt.ylabel("Precio")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(portfolio_values)
plt.title("Valor del portafolio durante la simulación")
plt.xlabel("Paso")
plt.ylabel("Valor del portafolio")
plt.grid(True)
plt.show()